# EDGE-FIRE full reviewed corpus — GPU
Requires all 600 prepared ZIPs (222.43 GiB), not the old 3,564-image pilot.
Checkpoints persist to your Google Drive. Authentication is performed by you.
This notebook starts a separate full-data GPU run; existing CPU checkpoints stay intact.

In [ ]:
import torch, shutil
assert torch.cuda.is_available(), "Select a GPU runtime; no CPU fallback"
print("GPU:", torch.cuda.get_device_name(0), "Torch:", torch.__version__)
print("Temporary disk free GiB:", round(shutil.disk_usage("/content").free / 2**30, 2))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import json, hashlib, shutil, sqlite3, subprocess, sys, os
# Upload the metadata/code package and all 600 ZIPs to this private folder.
PACKAGE = Path("/content/drive/MyDrive/EDGEFIRE/full_portable_20261122")
RUN = Path("/content/drive/MyDrive/EDGEFIRE/runs/full_yolo26n_gpu_v1")
assert (PACKAGE / "portable_manifest.json").is_file(), "Full metadata package has not been uploaded"
manifest = json.loads((PACKAGE / "portable_manifest.json").read_text())
with (PACKAGE / "full_dataset.sqlite").open("rb") as stream:
    assert hashlib.file_digest(stream, "sha256").hexdigest() == manifest["index_sha256"]
db = sqlite3.connect((PACKAGE / "full_dataset.sqlite").resolve().as_uri() + "?mode=ro", uri=True)
try:
    expected = list(db.execute("SELECT name,size,sha256 FROM shards ORDER BY name"))
finally:
    db.close()
paths = {}
for path in (PACKAGE / "shards").rglob("*.zip"):
    if path.name in paths:
        raise ValueError("Duplicate image ZIP: " + path.name)
    paths[path.name] = path
missing = [name for name,size,sha in expected if name not in paths or paths[name].stat().st_size != size]
print("Required ZIPs:",len(expected),"Missing/incomplete:",len(missing))
assert not missing, "Transfer all full-data ZIPs first. Missing examples: " + str(missing[:5])


In [ ]:
# Keep Colab's CUDA-enabled PyTorch; do not install the Windows CPU lockfile.
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.216","pyyaml"],check=True)


In [ ]:
# Cache complete ZIPs on temporary SSD to avoid random image reads over Drive.
# This is temporary training input; persistent checkpoints go to RUN on Drive.
LOCAL = Path("/content/edgefire_full_portable")
(LOCAL / "shards").mkdir(parents=True,exist_ok=True)
needed = sum(size for name,size,sha in expected if not (LOCAL/"shards"/name).is_file())
assert shutil.disk_usage("/content").free >= needed + 5*2**30, "Insufficient temporary disk for full corpus + 5 GiB reserve"
for name in ("full_dataset.sqlite","portable_manifest.json"):
    shutil.copyfile(PACKAGE/name,LOCAL/name)
shutil.copytree(PACKAGE/"code",LOCAL/"code",dirs_exist_ok=True)
for number,(name,size,sha) in enumerate(expected,1):
    target=LOCAL/"shards"/name
    valid=False
    if target.is_file() and target.stat().st_size==size:
        with target.open("rb") as stream:
            valid=hashlib.file_digest(stream,"sha256").hexdigest()==sha
    if not valid:
        temporary=target.with_suffix(".zip.partial")
        shutil.copyfile(paths[name],temporary)
        with temporary.open("rb") as stream:
            assert hashlib.file_digest(stream,"sha256").hexdigest()==sha, "Transfer corruption: "+name
        os.replace(temporary,target)
    if number%10==0 or number==len(expected):
        print("Verified image ZIPs:",number,"/",len(expected),flush=True)
print("Complete corpus cached; checkpoints will persist to",RUN)


In [ ]:
# No fraction/subset limit. Val uses all 79,218 retained validation images.
# Test remains untouched for final evaluation after model selection.
weights=PACKAGE/"pretrained/yolo26n-seg.pt"
assert weights.is_file(), "Registered generic pretrained checkpoint missing"
command=[sys.executable,"-u",str(LOCAL/"code/train_full_stream.py"),
    "--workspace",str(LOCAL),"--portable-data",str(LOCAL),
    "--run",str(RUN),"--device","cuda:0","--weights",str(weights),
    "--imgsz","512","--batch","8","--epochs","3","--threads","2"]
if (RUN/"last.pt").is_file():
    command.append("--resume")
print("Starting full-corpus GPU training. Inspect",RUN/"status.json",flush=True)
subprocess.run(command,check=True)
